In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [2]:
train_data = pd.read_csv('../data/model/full.csv')
train_labels = train_data['redemption_status'].values
train_data = train_data.drop(['id','customer_id','redemption_status'], axis=1)
valid_data = pd.read_csv('../data/model/valid.csv')
valid_labels = valid_data['redemption_status'].values
valid_data = valid_data.drop(['id','customer_id','redemption_status'], axis=1)
print(train_data.shape, valid_data.shape)

(78369, 66) (22606, 66)


In [3]:
params = {}
params['label'] = train_labels
params['feature_name'] = list(train_data.columns)
train_matrix = lgb.Dataset(train_data.values, **params)
params = {}
params['label'] = valid_labels
params['feature_name'] = list(valid_data.columns)
valid_matrix = lgb.Dataset(valid_data.values, **params)

In [4]:
booster = {}
booster['boosting_type'] = 'gbdt'
booster['objective'] = 'binary'
booster['learning_rate'] = 0.01
booster['num_leaves'] = 24
booster['max_depth'] = 4
booster['max_bin'] = 256
booster['subsample'] = 0.5
booster['subsample_freq'] = 1
booster['colsample_bylevel'] = 0.5
booster['colsample_bytree'] = 0.5
booster['min_split_gain'] = 0.0
booster['min_sum_hessian'] = 1
booster['nthread'] = 3
booster['verbose'] = 0
booster['metric'] = 'auc'

In [5]:
params = {}
params['params'] = booster
params['train_set'] = train_matrix
params['valid_sets'] = [train_matrix, valid_matrix]
params['num_boost_round'] = 700
params['early_stopping_rounds'] = 700
params['verbose_eval'] = 25

In [6]:
model = lgb.train(**params)

Training until validation scores don't improve for 700 rounds
[25]	training's auc: 0.959382	valid_1's auc: 0.961394
[50]	training's auc: 0.964504	valid_1's auc: 0.966174
[75]	training's auc: 0.968072	valid_1's auc: 0.969853
[100]	training's auc: 0.970335	valid_1's auc: 0.972246
[125]	training's auc: 0.972135	valid_1's auc: 0.973992
[150]	training's auc: 0.973855	valid_1's auc: 0.975486
[175]	training's auc: 0.975565	valid_1's auc: 0.977332
[200]	training's auc: 0.976715	valid_1's auc: 0.97856
[225]	training's auc: 0.977664	valid_1's auc: 0.979661
[250]	training's auc: 0.97854	valid_1's auc: 0.980593
[275]	training's auc: 0.979512	valid_1's auc: 0.981661
[300]	training's auc: 0.980236	valid_1's auc: 0.982403
[325]	training's auc: 0.980956	valid_1's auc: 0.983101
[350]	training's auc: 0.981625	valid_1's auc: 0.983675
[375]	training's auc: 0.982253	valid_1's auc: 0.984273
[400]	training's auc: 0.982885	valid_1's auc: 0.984843
[425]	training's auc: 0.983474	valid_1's auc: 0.985357
[450]	tr

In [7]:
model.save_model('../data/model/lightgbm_v1.model')

In [8]:
importance = model.feature_importance(importance_type='gain')
importance = pd.DataFrame(importance, columns=['importance'])
importance['feature'] = list(valid_data.columns)
importance['importance'] = importance['importance'] / importance['importance'].max()
importance = importance[['feature', 'importance']]
importance = importance.sort_values(by='importance', ascending=False)
importance = importance.reset_index(drop=True)

In [9]:
importance.head(10)

,feature,importance
0,cust_cdsc_cnt,1.000000
1,cust_coup_prc,0.980711
2,cust_cdsc_sum,0.636912
3,sum_trx_cust_coup_price,0.461444
4,cnt_coup_cdsc,0.395523
5,over_1,0.368327
6,max_trx_cust_coup_price,0.351856
7,sum_trx_cust_coup_qty,0.350586
8,cust_coup_cdsc,0.348508
9,cust_cdsc,0.303523


In [10]:
score_data = pd.read_csv('../data/model/test.csv')
score_driver = score_data[['id']].copy()
score_data = score_data.drop(['id','customer_id'], axis=1)
model = lgb.Booster(model_file='../data/model/lightgbm_v1.model')
score_driver['redemption_status'] = model.predict(score_data)
score_driver.to_csv('../data/score/score_v1.csv', index=False)

In [11]:
score_driver.shape

(50226, 2)